# Mapeamento de áreas de arenização com Sentinel-2 e Random Forest

**Projeto final – Future Tech Leaders Brasil, Space Tech Bootcamp 2026**
Autora: Franqueline Monback Noschang

Objetivo: mapear os areais existentes em 2025 na ottobacia Miracatu-Taquari (cód. 682636, entre Manoel Viana e São Francisco de Assis, RS), em 10 m e com acurácia avaliada.

| Etapa | O que faz |
|---|---|
| 1 | Configuração e autenticação no Earth Engine |
| 2 | Área de estudo (ottobacia 682636, ANA) |
| 3 | Coleção Sentinel-2 L2A de 2025 com máscara de nuvens |
| 4 | Índices por cena (NDVI, BSI, NDWI) e resumo anual por pixel |
| 5 | Amostras rotuladas (GeoJSON) e extração das variáveis |
| 6 | Treino do Random Forest (70%) com ajuste de hiperparâmetros e validação cruzada |
| 7 | Avaliação no teste (30%): matriz de confusão, acurácia global, precisão, recall, F1 e importância por classe |
| 8 | Classificação da ottobacia, área por classe e comparação com o MapBiomas 10 m |
| 9 | Exportação do mapa e dos polígonos de areais |
| 10 | Mapa web para o GitHub Pages (`docs/index.html`) |

**Antes de rodar, você precisa ter:**

1. Uma conta no Google Earth Engine vinculada a um projeto do Google Cloud.
2. O limite da ottobacia em `data/raw/area_de_estudo.shp` (já incluído).
3. As amostras rotuladas em `data/samples/amostras_2025.geojson` (a etapa 5 explica o formato). As etapas 1 a 4 rodam sem amostras.

## 1. Configuração

Rode a célula de instalação só uma vez, no ambiente virtual do projeto (`.venv`).

In [ ]:
# Instalação: rode uma vez no terminal, na raiz do repositório.
#   uv sync
# Isso cria o .venv com todas as bibliotecas do pyproject.toml.
# Depois selecione esse .venv como kernel do notebook no VS Code.
# Sem uv:
# %pip install -r ../requirements.txt

In [ ]:
import ee
import geemap
import geopandas as gpd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ID do seu projeto no Google Cloud (aparece no console do Earth Engine)
PROJETO_GCP = "SEU-PROJETO-GCP"

# Na primeira vez, abre o navegador para você autorizar o acesso.
# Depois disso, a autorização fica salva e esta linha pode ser comentada.
ee.Authenticate()
ee.Initialize(project=PROJETO_GCP)
print("Earth Engine conectado.")

In [ ]:
# ---------------- Parâmetros do projeto (ajuste aqui) ----------------
ANO = 2025

# Caminhos relativos à pasta notebooks/
# Ottobacia Miracatu-Taquari (cod_otto 682636, nível 6, 588 km²), entre Manoel Viana e São Francisco de Assis
CAMINHO_BACIA = Path("../data/raw/area_de_estudo.shp")
CAMINHO_AMOSTRAS = Path("../data/samples/amostras_2025.geojson")
PASTA_PROCESSADOS = Path("../data/processed")
PASTA_FIGURAS = Path("../figures")
PASTA_WEB = Path("../docs")
for pasta in (PASTA_PROCESSADOS, PASTA_FIGURAS, PASTA_WEB):
    pasta.mkdir(parents=True, exist_ok=True)

# Sistema de referência das saídas: SIRGAS 2000 / UTM 21S.
# A bacia inteira (55,46° a 55,01° O) está no fuso 21. Use o mesmo CRS nos critérios do AHP.
CRS_SAIDA = "EPSG:31981"
ESCALA = 10  # metros

# Máscara de nuvens: pixels com Cloud Score+ abaixo deste valor são descartados (0 a 1).
LIMIAR_NUVEM = 0.60

# Regra de "solo exposto" em cada cena: NDVI baixo e BSI positivo.
LIMIAR_NDVI_EXPOSTO = 0.25

# Classes do mapa. O código (número) é o valor do pixel no GeoTIFF final.
CLASSES = {1: "Areal", 2: "Campo", 3: "Lavoura", 4: "Silvicultura", 5: "Água"}
CORES = {1: "#f2d16b", 2: "#9ccf7a", 3: "#e974ed", 4: "#7a5c3a", 5: "#2f6fd6"}

# Divisão das amostras: 70% treino, 30% teste (separadas por polígono).
FRACAO_TESTE = 0.30

# MapBiomas 10 m (Sentinel-2): apoio à amostragem e comparação do mapa final.
# Cole aqui o ID do asset da Coleção 4 (10 m) publicado pelo MapBiomas, em
# brasil.mapbiomas.org > Downloads/Google Earth Engine. Deixe "" para pular essa parte.
# A coleção beta antiga (2016–2022) é:
#   "projects/mapbiomas-workspace/public/collection_S2_beta/collection_LULC_S2_beta"
ASSET_MAPBIOMAS_10M = ""
ANO_MAPBIOMAS = 2025  # use o ano mais recente disponível no asset

SEMENTE = 42  # garante que os sorteios sejam sempre os mesmos (reprodutibilidade)

## 2. Área de estudo

O limite da ottobacia é lido com `geopandas` e enviado ao Earth Engine com `geemap.gdf_to_ee`. O Earth Engine trabalha em coordenadas geográficas (EPSG:4326), por isso a reprojeção.

In [ ]:
bacia_gdf = gpd.read_file(CAMINHO_BACIA).to_crs("EPSG:4326")
bacia_gdf = bacia_gdf.dissolve()  # junta todas as partes num único polígono

bacia = geemap.gdf_to_ee(bacia_gdf)
geom = bacia.geometry()

area_km2 = geom.area(maxError=1).divide(1e6).getInfo()
print(f"Área da ottobacia: {area_km2:,.1f} km²")  # esperado: ~588 km² (campo area_total da ANA)

## 3. Coleção Sentinel-2 de 2025

- `COPERNICUS/S2_SR_HARMONIZED`: Sentinel-2 L2A, reflectância de superfície (já com correção atmosférica), ESA/Copernicus.
- `GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED`: nota de 0 a 1 por pixel, quanto maior, mais limpo. Usada como máscara de nuvens.

Em vez de usar uma imagem só, usamos **todas as cenas do ano**. É isso que permite medir o comportamento de cada pixel ao longo do tempo e separar o areal da lavoura em preparo de solo, que numa única data têm resposta espectral parecida.

In [ ]:
BANDAS = ["B2", "B3", "B4", "B8", "B11", "B12"]  # azul, verde, vermelho, NIR, SWIR1, SWIR2

s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(geom)
    .filterDate(f"{ANO}-01-01", f"{ANO + 1}-01-01")
)
cloud_score = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")


def preparar_cena(img):
    # Remove nuvens e sombras e converte os valores para reflectância (0 a 1).
    limpo = img.updateMask(img.select("cs_cdf").gte(LIMIAR_NUVEM))
    return (
        limpo.select(BANDAS)
        .multiply(0.0001)
        .copyProperties(img, ["system:time_start"])
    )


# linkCollection junta a nota de nuvem a cada cena correspondente
colecao = s2.linkCollection(cloud_score, ["cs_cdf"]).map(preparar_cena)
print("Cenas Sentinel-2 em", ANO, ":", colecao.size().getInfo())

## 4. Índices e variáveis anuais

**Por cena**, calculamos três índices:

- **NDVI** = (NIR − Vermelho) / (NIR + Vermelho): vegetação.
- **BSI** = ((SWIR1 + Vermelho) − (NIR + Azul)) / ((SWIR1 + Vermelho) + (NIR + Azul)): solo exposto.
- **NDWI** = (Verde − NIR) / (Verde + NIR): água.

**Por pixel, no ano todo**, resumimos as cenas nas variáveis do modelo:

| Variável | O que é |
|---|---|
| `B2_med` … `B12_med` | mediana anual das bandas azul, verde, vermelho, NIR, SWIR1 e SWIR2 |
| `NDVI_min` | NDVI mínimo do ano |
| `NDVI_medio` | NDVI médio do ano |
| `FREQ_EXPOSTO` | fração das cenas em que o pixel aparece como solo exposto (NDVI < 0,25 e BSI > 0) |

`FREQ_EXPOSTO` é a variável que separa o areal da lavoura: o **areal** fica exposto o ano inteiro (frequência alta); a **lavoura** só no preparo do solo (frequência intermediária). BSI e NDWI entram no cálculo de `FREQ_EXPOSTO` e nos mapas de apoio.

> **Atenção:** o NDVI mínimo é sensível a nuvem ou sombra que escapou da máscara. Confira a camada `NDVI_min` no mapa abaixo; se aparecerem manchas de nuvem, aumente `LIMIAR_NUVEM`.
>
> **Para o mestrado:** não incluímos declividade, solo ou geologia como variáveis. Assim o inventário de areais fica independente dos critérios do AHP e pode validá-lo sem circularidade.

In [ ]:
def adicionar_indices(img):
    ndvi = img.normalizedDifference(["B8", "B4"]).rename("NDVI")
    ndwi = img.normalizedDifference(["B3", "B8"]).rename("NDWI")
    bsi = img.expression(
        "((S1 + R) - (N + B)) / ((S1 + R) + (N + B))",
        {
            "S1": img.select("B11"),
            "R": img.select("B4"),
            "N": img.select("B8"),
            "B": img.select("B2"),
        },
    ).rename("BSI")
    # 1 = pixel com solo exposto nesta data, 0 = não
    exposto = ndvi.lt(LIMIAR_NDVI_EXPOSTO).And(bsi.gt(0)).rename("EXPOSTO").toFloat()
    return img.addBands([ndvi, ndwi, bsi, exposto])


colecao_idx = colecao.map(adicionar_indices)

# Mediana das bandas no ano (a "foto típica" do pixel)
medianas = colecao_idx.select(BANDAS).median().rename([f"{b}_med" for b in BANDAS])

# NDVI mínimo e médio do ano
ndvi_min = colecao_idx.select("NDVI").min().rename("NDVI_min")
ndvi_medio = colecao_idx.select("NDVI").mean().rename("NDVI_medio")

# Variável-chave: fração das observações em que o pixel apareceu exposto (0 a 1)
freq_exposto = colecao_idx.select("EXPOSTO").mean().rename("FREQ_EXPOSTO")

# Camadas de apoio (não entram no modelo)
bsi_med = colecao_idx.select("BSI").median().rename("BSI_med")
ndwi_med = colecao_idx.select("NDWI").median().rename("NDWI_med")
n_obs = colecao_idx.select("NDVI").count().rename("N_OBS")  # controle de qualidade

variaveis = ee.Image.cat([medianas, ndvi_min, ndvi_medio, freq_exposto]).toFloat().clip(geom)

VARIAVEIS = variaveis.bandNames().getInfo()
print(len(VARIAVEIS), "variáveis:", VARIAVEIS)

### Teste visual

Este é o **teste de capacidade** que combinamos: se o mapa abaixo aparecer, o ambiente está pronto. Na camada `FREQ_EXPOSTO`, os areais devem aparecer em amarelo-claro (exposto quase o ano todo). Compare com a imagem de satélite do fundo e tente achar alguns.

In [ ]:
mapa = geemap.Map()
mapa.add_basemap("SATELLITE")
mapa.centerObject(bacia, 12)

mapa.addLayer(
    variaveis,
    {"bands": ["B4_med", "B3_med", "B2_med"], "min": 0.02, "max": 0.18},
    "Sentinel-2 cor natural (mediana 2025)",
)
mapa.addLayer(
    variaveis.select("NDVI_min"),
    {"min": -0.1, "max": 0.6, "palette": ["#a6611a", "#f5f5f5", "#1a9641"]},
    "NDVI mínimo",
    False,
)
mapa.addLayer(
    variaveis.select("NDVI_medio"),
    {"min": 0, "max": 0.8, "palette": ["#a6611a", "#f5f5f5", "#1a9641"]},
    "NDVI médio",
    False,
)
mapa.addLayer(bsi_med.clip(geom), {"min": -0.3, "max": 0.3, "palette": ["#1a9641", "#f5f5f5", "#a6611a"]}, "BSI (mediana)", False)
mapa.addLayer(ndwi_med.clip(geom), {"min": -0.6, "max": 0.3, "palette": ["#f5f5f5", "#2f6fd6"]}, "NDWI (mediana)", False)
mapa.addLayer(
    variaveis.select("FREQ_EXPOSTO"),
    {"min": 0, "max": 1, "palette": ["#1b1b1b", "#7f3b08", "#fdb863", "#fff7bc"]},
    "Frequência de solo exposto",
)
mapa.addLayer(n_obs.clip(geom), {"min": 0, "max": 60, "palette": ["red", "yellow", "green"]}, "Nº de observações", False)

# MapBiomas 10 m como apoio à amostragem (se o asset estiver configurado)
if ASSET_MAPBIOMAS_10M:
    mb = ee.Image(ASSET_MAPBIOMAS_10M).select(f"classification_{ANO_MAPBIOMAS}").clip(geom)
    mapa.addLayer(mb.eq(23).selfMask(), {"palette": ["#ff0000"]}, "MapBiomas: praia, duna e areal (23)", False)

mapa.addLayer(bacia.style(color="white", fillColor="00000000", width=2), {}, "Ottobacia")
mapa

## 5. Amostras rotuladas

**Formato:** `data/samples/amostras_2025.geojson`, em qualquer CRS, com os campos:

| Campo | Tipo | Conteúdo |
|---|---|---|
| `classe` | inteiro | 1 Areal, 2 Campo, 3 Lavoura, 4 Silvicultura, 5 Água |
| `poligono` | inteiro | número único de cada polígono (usado para separar treino e teste) |

**Como criar no QGIS:**

1. Crie uma camada de **polígonos** e salve como GeoJSON em `data/samples/amostras_2025.geojson`.
2. Desenhe polígonos pequenos e "puros" sobre cada classe, por interpretação visual sobre imagens de alta resolução (Google Satellite), usando a mediana Sentinel-2 e o MapBiomas 10 m como apoio.
3. Meta: **100 a 200 amostras (pontos) por classe**, vindas de polígonos bem espalhados pela bacia. O código sorteia até `PONTOS_POR_POLIGONO_MAX` pontos dentro de cada polígono; a tabela abaixo mostra quantos pontos cada classe recebeu.

**Por que polígonos e não pontos soltos?** Pixels vizinhos são quase idênticos. Se um ficar no treino e o vizinho no teste, a acurácia sai inflada. Separando treino e teste **por polígono**, a avaliação fica honesta (autocorrelação espacial). Se você preferir pontos soltos, o código também aceita: cada ponto vira o seu próprio grupo.

In [ ]:
PONTOS_POR_POLIGONO_MAX = 20


def pontos_em_poligonos(poligonos_gdf, max_pontos, semente=SEMENTE):
    # Sorteia pontos dentro de cada polígono, proporcional à área (no máximo max_pontos).
    # Cada ponto herda 'classe' e 'poligono'.
    proj = poligonos_gdf.to_crs(CRS_SAIDA)
    pixels_por_poligono = (proj.area / (ESCALA * ESCALA)).astype(int)
    n_pontos = np.clip(pixels_por_poligono // 2, 3, max_pontos)
    partes = []
    for (idx, linha), n in zip(proj.iterrows(), n_pontos):
        amostra = gpd.GeoSeries([linha.geometry], crs=proj.crs).sample_points(size=int(n), rng=semente + int(idx))
        pts = amostra.explode(index_parts=False)
        partes.append(gpd.GeoDataFrame(
            {"classe": int(linha["classe"]), "poligono": int(linha["poligono"])},
            index=range(len(pts)), geometry=pts.values, crs=proj.crs,
        ))
    return pd.concat(partes, ignore_index=True).to_crs("EPSG:4326")


amostras_gdf = gpd.read_file(CAMINHO_AMOSTRAS)
if amostras_gdf.geom_type.isin(["Polygon", "MultiPolygon"]).all():
    pontos_gdf = pontos_em_poligonos(amostras_gdf, PONTOS_POR_POLIGONO_MAX)
else:
    # Pontos soltos: cada ponto vira o seu próprio "polígono" na validação
    pontos_gdf = amostras_gdf.to_crs("EPSG:4326").copy()
    if "poligono" not in pontos_gdf.columns:
        print("Aviso: sem coluna 'poligono'; a validação ficará otimista por causa de pontos vizinhos.")
        pontos_gdf["poligono"] = range(len(pontos_gdf))

resumo = (pontos_gdf.groupby("classe").agg(pontos=("classe", "size"), poligonos=("poligono", "nunique"))
          .rename(index=CLASSES))
print(resumo)
fora_da_meta = resumo[(resumo["pontos"] < 100) | (resumo["pontos"] > 200)]
if len(fora_da_meta):
    print("\nAviso: classes fora da meta de 100–200 amostras:", list(fora_da_meta.index))

In [ ]:
# Envia os pontos ao Earth Engine e lê o valor de todas as variáveis em cada um
pontos_ee = geemap.gdf_to_ee(pontos_gdf[["classe", "poligono", "geometry"]])
amostrado = variaveis.sampleRegions(
    collection=pontos_ee,
    properties=["classe", "poligono"],
    scale=ESCALA,
    tileScale=4,
)

df = geemap.ee_to_df(amostrado)
df = df.dropna(subset=VARIAVEIS)  # descarta pontos sem imagem válida (ex.: borda da bacia)
df[["classe", "poligono"]] = df[["classe", "poligono"]].astype(int)
df.to_csv(PASTA_PROCESSADOS / f"amostras_variaveis_{ANO}.csv", index=False)
print(df.shape[0], "amostras com variáveis.")
df.groupby("classe")[["FREQ_EXPOSTO", "NDVI_min", "NDVI_medio"]].median().rename(index=CLASSES).round(2)

A tabela acima já é uma **análise exploratória** útil para a apresentação: se a mediana de `FREQ_EXPOSTO` do areal for bem maior que a da lavoura, a variável está fazendo o trabalho dela.

## 6. Treino do Random Forest

1. **Divisão 70/30:** separamos 30% das amostras para teste e 70% para treino. A separação é feita **por polígono** e mantém a proporção das classes (`StratifiedGroupKFold` com 10 dobras: 3 dobras viram teste, 7 viram treino). O teste só é usado na etapa 7.
2. **Ajuste de hiperparâmetros:** nos 70% de treino, testamos combinações de **número de árvores**, **profundidade máxima** e **variáveis por divisão** com validação cruzada de 5 dobras (também por polígono).
3. A métrica de escolha é o **F1 macro**, a média do F1 de cada classe. Ela impede que o modelo "ganhe" acertando só as classes grandes (campo) e errando a pequena (areal).

**Por que Random Forest?** É uma classificação de pixels com poucas amostras rotuladas (centenas) e variáveis tabulares. O RF funciona bem nesse cenário, é robusto a variáveis correlacionadas, mostra quais variáveis importam e roda nativamente no Earth Engine.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold

METRICA = "f1_macro"

X = df[VARIAVEIS].to_numpy()
y = df["classe"].to_numpy()
grupos = df["poligono"].to_numpy()

# Divisão 70/30 por polígono: 10 dobras estratificadas, 3 delas formam o teste
N_DOBRAS = 10
dobra = np.empty(len(y), dtype=int)
divisor = StratifiedGroupKFold(n_splits=N_DOBRAS, shuffle=True, random_state=SEMENTE)
for k, (_, idx) in enumerate(divisor.split(X, y, grupos)):
    dobra[idx] = k
eh_teste = dobra < round(FRACAO_TESTE * N_DOBRAS)

X_treino, y_treino, g_treino = X[~eh_teste], y[~eh_teste], grupos[~eh_teste]
X_teste, y_teste = X[eh_teste], y[eh_teste]
print(f"Treino: {len(y_treino)} pontos ({(~eh_teste).mean():.0%}) | "
      f"Teste: {len(y_teste)} pontos ({eh_teste.mean():.0%})")
print(pd.crosstab(df["classe"].map(CLASSES), np.where(eh_teste, "teste", "treino")))

In [ ]:
# Ajuste de hiperparâmetros com validação cruzada espacial (5 dobras por polígono)
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEMENTE)

grade = {
    "n_estimators": [100, 300, 500],      # número de árvores
    "max_depth": [None, 10, 20],          # profundidade máxima (None = sem limite)
    "max_features": ["sqrt", 0.5],        # variáveis sorteadas em cada divisão
}

busca = GridSearchCV(
    RandomForestClassifier(random_state=SEMENTE, n_jobs=-1),
    grade, cv=cv, scoring=METRICA, n_jobs=-1,
)
busca.fit(X_treino, y_treino, groups=g_treino)

resultados = (pd.DataFrame(busca.cv_results_)
              [["param_n_estimators", "param_max_depth", "param_max_features", "mean_test_score", "std_test_score"]]
              .sort_values("mean_test_score", ascending=False))
resultados.to_csv(PASTA_PROCESSADOS / "ajuste_hiperparametros_rf.csv", index=False)

print(f"Melhores hiperparâmetros: {busca.best_params_}")
print(f"F1 macro na validação cruzada: {busca.best_score_:.3f} "
      f"± {busca.cv_results_['std_test_score'][busca.best_index_]:.3f}")
resultados.head(10)

## 7. Avaliação no conjunto de teste (30%)

Métricas: matriz de confusão, acurácia global, precisão, recall e F1 por classe, e importância das variáveis por classe. Como referência, Rayne et al. (2023) obtiveram 74–76% de acurácia na classe de desertificação com Sentinel-2 e RF.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score

modelo = busca.best_estimator_
y_previsto = modelo.predict(X_teste)

codigos = sorted(CLASSES)
nomes = [CLASSES[c] for c in codigos]

print(f"Acurácia global (teste): {accuracy_score(y_teste, y_previsto):.3f}")
print(f"F1 macro (teste): {f1_score(y_teste, y_previsto, labels=codigos, average='macro'):.3f}\n")

relatorio = classification_report(y_teste, y_previsto, labels=codigos, target_names=nomes, digits=3, output_dict=True)
pd.DataFrame(relatorio).T.to_csv(PASTA_PROCESSADOS / f"metricas_teste_{ANO}.csv")
print(classification_report(y_teste, y_previsto, labels=codigos, target_names=nomes, digits=3))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_teste, y_previsto, labels=codigos, display_labels=nomes,
    cmap="Blues", colorbar=False, ax=ax,
)
ax.set_xlabel("Classe prevista")
ax.set_ylabel("Classe de referência")
ax.set_title(f"Matriz de confusão – teste ({ANO})")
plt.tight_layout()
fig.savefig(PASTA_FIGURAS / "matriz_confusao.png", dpi=200)
plt.show()

In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.metrics import make_scorer

# Importância por classe: quanto o F1 de cada classe cai quando embaralhamos cada variável
# (importância por permutação, calculada no conjunto de teste).
importancia = {}
for c in codigos:
    f1_da_classe = make_scorer(f1_score, labels=[c], average="macro", zero_division=0)
    imp = permutation_importance(
        modelo, X_teste, y_teste, scoring=f1_da_classe, n_repeats=20, random_state=SEMENTE, n_jobs=-1,
    )
    importancia[CLASSES[c]] = imp.importances_mean

importancia = pd.DataFrame(importancia, index=VARIAVEIS)
importancia["Média"] = importancia.mean(axis=1)
importancia = importancia.sort_values("Média", ascending=False)
importancia.to_csv(PASTA_PROCESSADOS / f"importancia_por_classe_{ANO}.csv")

fig, ax = plt.subplots(figsize=(7, 4.5))
im = ax.imshow(importancia[nomes].to_numpy(), cmap="Blues", aspect="auto")
ax.set_xticks(range(len(nomes)), nomes)
ax.set_yticks(range(len(importancia)), importancia.index)
for i in range(len(importancia)):
    for j, n in enumerate(nomes):
        ax.text(j, i, f"{importancia.iloc[i][n]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax, label="Queda média no F1 da classe")
ax.set_title("Importância das variáveis por classe")
plt.tight_layout()
fig.savefig(PASTA_FIGURAS / "importancia_por_classe.png", dpi=200)
plt.show()

## 8. Classificação da ottobacia

O mapa é gerado pelo Random Forest do próprio Earth Engine (`smileRandomForest`), com os hiperparâmetros escolhidos na etapa 6 e treinado com os mesmos 70% de treino. Assim o processamento da bacia inteira roda nos servidores do Google.

Equivalência dos hiperparâmetros: `n_estimators` → `numberOfTrees`; `max_features` → `variablesPerSplit`; `max_depth` → `maxNodes` (o GEE limita o número de folhas, e uma árvore de profundidade *d* tem no máximo 2^*d* folhas).

No fim conferimos a acurácia desse modelo nos mesmos 30% de teste. Ela deve ficar próxima da obtida no scikit-learn.

In [ ]:
melhores = busca.best_params_
n_var = len(VARIAVEIS)
if melhores["max_features"] == "sqrt":
    var_por_divisao = int(round(np.sqrt(n_var)))
else:
    var_por_divisao = int(round(melhores["max_features"] * n_var))
max_folhas = None if melhores["max_depth"] is None else 2 ** int(melhores["max_depth"])

poligonos_treino = sorted({int(g) for g in g_treino})
treino_ee = amostrado.filter(ee.Filter.inList("poligono", poligonos_treino))
teste_ee = amostrado.filter(ee.Filter.inList("poligono", poligonos_treino).Not())

classificador = ee.Classifier.smileRandomForest(
    numberOfTrees=int(melhores["n_estimators"]),
    variablesPerSplit=var_por_divisao,
    maxNodes=max_folhas,
    seed=SEMENTE,
).train(features=treino_ee, classProperty="classe", inputProperties=VARIAVEIS)

mapa_classes = variaveis.classify(classificador).rename("classe").toByte().clip(geom)

# Conferência no teste
matriz_ee = teste_ee.classify(classificador).errorMatrix("classe", "classification")
print(f"Acurácia global (GEE, teste): {matriz_ee.accuracy().getInfo():.3f}")
print(f"Kappa (GEE, teste): {matriz_ee.kappa().getInfo():.3f}")

In [ ]:
# Área de cada classe na ottobacia (km²)
area_por_classe = (
    ee.Image.pixelArea().divide(1e6)
    .addBands(mapa_classes)
    .reduceRegion(
        reducer=ee.Reducer.sum().group(groupField=1, groupName="classe"),
        geometry=geom, scale=ESCALA, maxPixels=1e10,
    )
    .get("groups")
    .getInfo()
)
tabela_area = pd.DataFrame(area_por_classe).rename(columns={"sum": "area_km2"})
tabela_area["classe_nome"] = tabela_area["classe"].map(CLASSES)
tabela_area["percentual"] = 100 * tabela_area["area_km2"] / tabela_area["area_km2"].sum()
tabela_area.to_csv(PASTA_PROCESSADOS / f"area_por_classe_{ANO}.csv", index=False)
tabela_area.round(2)

> A área "contada no mapa" ainda carrega os erros do classificador. Para a dissertação, vale corrigi-la com a matriz de confusão, pelo método de Olofsson et al. (2014). Para o capstone, a tabela acima já basta.

In [ ]:
mapa2 = geemap.Map()
mapa2.add_basemap("SATELLITE")
mapa2.centerObject(bacia, 12)
mapa2.addLayer(
    mapa_classes,
    {"min": min(codigos), "max": max(codigos), "palette": [CORES[c].lstrip("#") for c in codigos]},
    f"Classificação {ANO}",
)
mapa2.addLayer(bacia.style(color="white", fillColor="00000000", width=2), {}, "Ottobacia")
mapa2.add_legend(title="Classes", legend_dict={CLASSES[c]: CORES[c] for c in codigos})
mapa2

### Comparação com o MapBiomas 10 m

As classes do MapBiomas são agrupadas nas cinco classes do projeto (23 praia, duna e areal → Areal; 12 e 15 → Campo; agricultura → Lavoura; 9 → Silvicultura; 31 e 33 → Água) e cruzadas com o nosso mapa. A tabela mostra, em km², onde os dois concordam e onde o nosso mapa separa o areal de outras coberturas que o MapBiomas agrupa de outra forma. Esta etapa só roda se `ASSET_MAPBIOMAS_10M` estiver preenchido.

In [ ]:
if ASSET_MAPBIOMAS_10M:
    # Códigos da legenda MapBiomas -> classes do projeto (0 = fora das cinco classes)
    MB_PARA_PROJETO = {
        23: 1,                                                        # praia, duna e areal
        12: 2, 15: 2,                                                 # campo e pastagem
        18: 3, 19: 3, 20: 3, 35: 3, 36: 3, 39: 3, 40: 3, 41: 3,
        46: 3, 47: 3, 48: 3, 62: 3,                                   # agricultura
        9: 4,                                                         # silvicultura
        31: 5, 33: 5,                                                 # água
    }
    mb = ee.Image(ASSET_MAPBIOMAS_10M).select(f"classification_{ANO_MAPBIOMAS}")
    mb_proj = mb.remap(list(MB_PARA_PROJETO), list(MB_PARA_PROJETO.values()), 0).rename("mapbiomas")

    # Código combinado: 10 × classe do projeto + classe MapBiomas agrupada
    cruzado = mapa_classes.multiply(10).add(mb_proj).rename("par")
    grupos_area = (
        ee.Image.pixelArea().divide(1e6).addBands(cruzado)
        .reduceRegion(
            reducer=ee.Reducer.sum().group(groupField=1, groupName="par"),
            geometry=geom, scale=ESCALA, maxPixels=1e10,
        )
        .get("groups").getInfo()
    )
    nomes_mb = {**CLASSES, 0: "Outras"}
    cruz = pd.DataFrame(grupos_area)
    cruz["Este projeto"] = (cruz["par"] // 10).map(CLASSES)
    cruz["MapBiomas 10 m"] = (cruz["par"] % 10).map(nomes_mb)
    tabela_mb = cruz.pivot_table(index="Este projeto", columns="MapBiomas 10 m", values="sum", aggfunc="sum", fill_value=0)
    tabela_mb.to_csv(PASTA_PROCESSADOS / f"comparacao_mapbiomas_{ANO}.csv")
    display(tabela_mb.round(2))
else:
    print("ASSET_MAPBIOMAS_10M vazio: comparação com o MapBiomas pulada.")

## 9. Exportação para o QGIS

As exportações vão para uma pasta `arenizacao` no seu Google Drive (acompanhe na aba *Tasks* do Code Editor ou com `tarefa.status()`):

- `classes_2025.tif`: o mapa (1 = areal, 2 = campo, 3 = lavoura, 4 = silvicultura, 5 = água), em SIRGAS 2000 / UTM, 10 m.
- `variaveis_2025.tif`: todas as variáveis. `FREQ_EXPOSTO` e `NDVI_p10` podem virar camadas de apoio no mestrado.
- `areais_2025.shp`: os areais vetorizados, com a área de cada polígono em hectares. Use para validar o AHP depois.

In [ ]:
AREA_MIN_HA = 0.1  # descarta manchas menores que ~10 pixels (ruído)

tarefas = []

tarefas.append(ee.batch.Export.image.toDrive(
    image=mapa_classes,
    description=f"classes_{ANO}",
    folder="arenizacao",
    region=geom,
    scale=ESCALA,
    crs=CRS_SAIDA,
    maxPixels=1e10,
    fileFormat="GeoTIFF",
))

tarefas.append(ee.batch.Export.image.toDrive(
    image=variaveis,
    description=f"variaveis_{ANO}",
    folder="arenizacao",
    region=geom,
    scale=ESCALA,
    crs=CRS_SAIDA,
    maxPixels=1e10,
    fileFormat="GeoTIFF",
))

areais_vetor = (
    mapa_classes.eq(1).selfMask()
    .reduceToVectors(
        geometry=geom, scale=ESCALA, crs=CRS_SAIDA,
        geometryType="polygon", eightConnected=True, maxPixels=1e10,
    )
    .map(lambda f: f.set("area_ha", f.geometry().area(maxError=1).divide(1e4)))
    .filter(ee.Filter.gte("area_ha", AREA_MIN_HA))
)
tarefas.append(ee.batch.Export.table.toDrive(
    collection=areais_vetor,
    description=f"areais_{ANO}",
    folder="arenizacao",
    fileFormat="SHP",
))

for tarefa in tarefas:
    tarefa.start()
    print("Exportação iniciada:", tarefa.config["description"])

## 10. Mapa web para o GitHub Pages

Gera `docs/index.html`, um mapa interativo (Leaflet) com a classificação de 2025, os polígonos de areais e o limite da ottobacia. O mapa é estático, não depende do Earth Engine para abrir e fica publicado em https://franquinoschang.github.io/arenizacao-pampa/ depois do `git push`.

In [ ]:
import urllib.request

# 1) Imagem PNG da classificação (Web Mercator, para alinhar com o Leaflet)
regiao = geom.bounds()
cantos = regiao.coordinates().getInfo()[0]
oeste, leste = min(c[0] for c in cantos), max(c[0] for c in cantos)
sul, norte = min(c[1] for c in cantos), max(c[1] for c in cantos)

url_png = mapa_classes.visualize(
    min=min(codigos), max=max(codigos), palette=[CORES[c].lstrip("#") for c in codigos],
).getThumbURL({"region": regiao, "dimensions": 2048, "crs": "EPSG:3857", "format": "png"})
urllib.request.urlretrieve(url_png, PASTA_WEB / f"classes_{ANO}.png")

# 2) Polígonos de areais e limite da bacia em GeoJSON (WGS84), simplificados para a web
areais_gdf = geemap.ee_to_gdf(areais_vetor).to_crs("EPSG:4326")
areais_gdf["geometry"] = areais_gdf.geometry.simplify(0.00005)
areais_gdf["area_ha"] = areais_gdf["area_ha"].round(2)
areais_json = areais_gdf[["area_ha", "geometry"]].to_json()
bacia_json = bacia_gdf[["geometry"]].to_json()

# Aspas simples no HTML porque a legenda entra numa string JavaScript com aspas duplas
legenda = "".join(
    f"<div><span style='background:{CORES[c]}'></span>{CLASSES[c]}</div>" for c in codigos
)
area_areal = tabela_area.loc[tabela_area["classe"] == 1, "area_km2"].sum()

html = """<!doctype html>
<html lang="pt-BR">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Areais da ottobacia Miracatu-Taquari – __ANO__</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css">
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<style>
  html, body, #mapa { height: 100%; margin: 0; font-family: system-ui, sans-serif; }
  .painel { background: #fff; padding: 8px 12px; border-radius: 6px; box-shadow: 0 1px 4px #0004; font-size: 13px; line-height: 1.5; max-width: 260px; }
  .painel h1 { font-size: 15px; margin: 0 0 4px; }
  .painel span { display: inline-block; width: 14px; height: 14px; margin-right: 6px; vertical-align: -2px; border: 1px solid #0003; }
</style>
</head>
<body>
<div id="mapa"></div>
<script>
const mapa = L.map("mapa");
const satelite = L.tileLayer("https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
  { maxZoom: 18, attribution: "Imagem: Esri" }).addTo(mapa);
const osm = L.tileLayer("https://tile.openstreetmap.org/{z}/{x}/{y}.png",
  { maxZoom: 19, attribution: "&copy; OpenStreetMap" });
const limites = [[__SUL__, __OESTE__], [__NORTE__, __LESTE__]];
const classes = L.imageOverlay("classes___ANO__.png", limites, { opacity: 0.75 }).addTo(mapa);
const areais = L.geoJSON(__AREAIS__, {
  style: { color: "#d7301f", weight: 1.5, fillOpacity: 0 },
  onEachFeature: (f, l) => l.bindPopup("Areal: " + f.properties.area_ha + " ha"),
}).addTo(mapa);
const bacia = L.geoJSON(__BACIA__, { style: { color: "#fff", weight: 2, fill: false } }).addTo(mapa);
mapa.fitBounds(bacia.getBounds());
L.control.layers({ "Satélite": satelite, "OpenStreetMap": osm },
  { "Classificação __ANO__": classes, "Polígonos de areais": areais, "Ottobacia": bacia }).addTo(mapa);
const info = L.control({ position: "bottomleft" });
info.onAdd = () => {
  const d = L.DomUtil.create("div", "painel");
  d.innerHTML = "<h1>Areais – ottobacia Miracatu-Taquari (__ANO__)</h1>"
    + "Sentinel-2 + Random Forest · área de areal: <b>__AREA__ km²</b>"
    + "__LEGENDA__"
    + "<div style='margin-top:4px'><a href='https://github.com/Franquinoschang/arenizacao-pampa'>Código e metodologia</a></div>";
  return d;
};
info.addTo(mapa);
</script>
</body>
</html>
"""
substituicoes = {
    "__ANO__": str(ANO), "__SUL__": f"{sul:.6f}", "__NORTE__": f"{norte:.6f}",
    "__OESTE__": f"{oeste:.6f}", "__LESTE__": f"{leste:.6f}",
    "__AREAIS__": areais_json, "__BACIA__": bacia_json,
    "__AREA__": f"{area_areal:,.2f}".replace(",", "X").replace(".", ",").replace("X", "."),
    "__LEGENDA__": legenda,
}
for chave, valor in substituicoes.items():
    html = html.replace(chave, valor)
(PASTA_WEB / "index.html").write_text(html, encoding="utf-8")
print(f"Mapa web salvo em {PASTA_WEB / 'index.html'} ({len(areais_gdf)} polígonos de areais).")

## Próximos passos

- **Resultados no README:** preencher a seção de resultados com a acurácia global, o F1 do areal, a matriz de confusão, a importância por classe e a área de areal na ottobacia.
- **Comparação com a literatura:** comparar a acurácia da classe Areal com os 74–76% de Rayne et al. (2023) e discutir se as variáveis anuais reduziram a confusão entre areal e solo de cultivo relatada por Souza e Pires (2017) com imagem de data única.
- **Mestrado:** abrir `classes_2025.tif` e `areais_2025.shp` no QGIS, na mesma grade dos critérios do AHP, e usar o inventário para validar o mapa de suscetibilidade.
- **Extensão futura:** aplicar o modelo a anos anteriores (troque `ANO` e rode de novo as etapas 3, 4, 8 a 10) para medir a expansão dos areais.